# Week 1 — Problem Definition, EDA & Data Understanding
**Project:** BrandSphere AI | **Brand:** NovaTech | **Scenario:** 1

## Dataset Column Mapping
| File | Rows | Exact Columns | Module |
|---|---|---|---|
| marketing_campaign_dataset.csv | 200,000 | Campaign_ID, Company, Campaign_Type, Target_Audience, Duration, Channel_Used, Conversion_Rate, Acquisition_Cost, ROI, Location, Language, Clicks, Impressions, Engagement_Score, Customer_Segment, Date | Campaign prediction & analytics |
| sloganlist.csv | 1,162 | Company, Slogan | NLP tagline generation |
| startups.csv | 42,038 | name, city, tagline, description | Brand persona profiling |
| Logo Dataset (Kaggle) | 167,140 images | class-labelled folders | CNN classification |
| Font Dataset (Kaggle) | image files | class-labelled folders | KNN font engine |

In [ ]:
!pip install pandas numpy matplotlib seaborn plotly pillow -q
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px, re, json, warnings
from collections import Counter
warnings.filterwarnings('ignore')
BRAND = ['#1A1A1A','#5055D8','#0D1B3E','#C0C0C8','#F9F9F7']
print('Libraries loaded ✓')

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/BrandSphere/'
PATHS = {'marketing':BASE+'marketing_campaign_dataset.csv','slogans':BASE+'sloganlist.csv','startups':BASE+'startups.csv','logo_dir':BASE+'logo_dataset/','font_dir':BASE+'font_dataset/'}
import os
for k,v in PATHS.items(): print(f'  {k:12s}: {"✓" if os.path.exists(v) else "✗ check path"}')


In [ ]:
# Load and clean Marketing Dataset — 200,000 rows
mkt = pd.read_csv(PATHS['marketing'])
print('Shape:', mkt.shape)
print('Columns:', list(mkt.columns))
# Clean Acquisition_Cost: '$16,174.00' → 16174.0
mkt['Acquisition_Cost'] = mkt['Acquisition_Cost'].astype(str).str.replace('[$,]','',regex=True).astype(float)
# Duration: '30 days' → 30
mkt['Duration_Days'] = mkt['Duration'].str.extract(r'(\d+)').astype(int)
# Derived CTR
mkt['CTR'] = (mkt['Clicks'] / mkt['Impressions'] * 100).round(4)
print('\nNull values:', mkt.isnull().sum().sum())
print('Duplicates:', mkt.duplicated().sum())
display(mkt[['Conversion_Rate','Acquisition_Cost','ROI','CTR','Engagement_Score','Duration_Days']].describe())

In [ ]:
# Load Slogans (Company, Slogan) and Startups (name, city, tagline, description)
sl = pd.read_csv(PATHS['slogans']); sl.columns=['Company','Slogan']
sl['Slogan']=sl['Slogan'].astype(str).str.strip(); sl=sl.dropna()
print('Slogans:',len(sl),'| Columns:',list(sl.columns)); display(sl.head(3))
st_df=pd.read_csv(PATHS['startups']).dropna(subset=['tagline'])
print('\nStartups with taglines:',len(st_df),'| Columns:',list(st_df.columns)); display(st_df[['name','city','tagline']].head(3))

In [ ]:
# EDA Visualisations
fig,axes=plt.subplots(2,3,figsize=(18,9))
mkt['Campaign_Type'].value_counts().plot(kind='bar',ax=axes[0,0],color=BRAND[0]); axes[0,0].set_title('Campaign Types',fontweight='bold')
mkt['Channel_Used'].value_counts().plot(kind='bar',ax=axes[0,1],color=BRAND[1]); axes[0,1].set_title('Channels Used',fontweight='bold')
mkt['ROI'].hist(bins=30,ax=axes[0,2],color=BRAND[2],edgecolor='white'); axes[0,2].set_title('ROI Distribution',fontweight='bold')
mkt['CTR'].hist(bins=40,ax=axes[1,0],color=BRAND[1],edgecolor='white'); axes[1,0].set_title('CTR Distribution',fontweight='bold')
mkt['Engagement_Score'].hist(bins=10,ax=axes[1,1],color=BRAND[3],edgecolor='white'); axes[1,1].set_title('Engagement Score (1-10)',fontweight='bold')
mkt.groupby('Channel_Used')['CTR'].mean().sort_values().plot(kind='barh',ax=axes[1,2],color=BRAND[0]); axes[1,2].set_title('Avg CTR by Channel',fontweight='bold')
plt.suptitle('NovaTech — Marketing Dataset EDA (200,000 records)',fontsize=14,fontweight='bold',y=1.01)
plt.tight_layout(); plt.savefig('week1_eda.png',dpi=150,bbox_inches='tight'); plt.show(); print('Saved ✓')

In [ ]:
# Slogan word frequency
text=' '.join(sl['Slogan'].str.lower()); words=re.findall(r'\b[a-z]{4,}\b',text)
stop={'your','with','that','this','from','have','more','make','will','just','than','they','their','been','what','every','always'}
freq=Counter(w for w in words if w not in stop).most_common(20)
fig2,ax=plt.subplots(figsize=(12,5))
ax.barh([w for w,_ in freq],[c for _,c in freq],color='#5055D8')
ax.set_title('Top 20 Words in 1,162 Brand Slogans',fontweight='bold'); ax.set_xlabel('Frequency')
plt.tight_layout(); plt.savefig('week1_slogan_words.png',dpi=150); plt.show()

In [ ]:
# Correlation heatmap
num=['Conversion_Rate','Acquisition_Cost','ROI','Clicks','Impressions','Engagement_Score','CTR','Duration_Days']
plt.figure(figsize=(10,7))
sns.heatmap(mkt[num].corr(),annot=True,fmt='.2f',cmap='Blues',linewidths=.5,mask=np.triu(np.ones((len(num),len(num)),bool)))
plt.title('Marketing Dataset Correlation Matrix',fontweight='bold')
plt.tight_layout(); plt.savefig('week1_correlation.png',dpi=150); plt.show()

In [ ]:
# Save stats JSON for Streamlit app
stats={'total_campaigns':int(len(mkt)),'channels':mkt['Channel_Used'].value_counts().to_dict(),'campaign_types':mkt['Campaign_Type'].value_counts().to_dict(),'locations':mkt['Location'].value_counts().to_dict(),'segments':mkt['Customer_Segment'].value_counts().to_dict(),'avg_ctr':round(float(mkt['CTR'].mean()),4),'avg_roi':round(float(mkt['ROI'].mean()),4),'avg_engagement':round(float(mkt['Engagement_Score'].mean()),4)}
with open('week1_stats.json','w') as f: json.dump(stats,f,indent=2)
print('Saved week1_stats.json ✓'); print(json.dumps({k:v for k,v in list(stats.items())[:5]},indent=2))

## ✅ Week 1 Deliverables
- [ ] All datasets loaded with exact column names verified
- [ ] marketing_campaign_dataset.csv cleaned: Acquisition_Cost, Duration_Days, CTR derived
- [ ] Data quality: 0 nulls in key columns, 0 duplicates
- [ ] Visualisations: week1_eda.png, week1_slogan_words.png, week1_correlation.png
- [ ] Stats saved: week1_stats.json

### Key Findings
1. **Marketing**: CTR derived as Clicks/Impressions×100. ROI range 2–8, Engagement 1–10 (uniform — synthetic dataset)
2. **Slogans**: Company & Slogan columns. Top words: original, good, love, world, taste
3. **Startups**: name, city, tagline, description. 36,656 have taglines